# Ingestion météo — tous les POI

Appel de l'API météo sur l'ensemble des POI (50), pour couvrir
entièrement la base avec des données météo. Chaque appel utilise
directement les coordonnées du POI, ce qui rattache chaque point
météo à un poi_id exact dès l'ingestion (voir ADR-009).

In [1]:
import logging
import os
from pathlib import Path

import boto3
import polars as pl
from dotenv import load_dotenv

from ingestion.meteo import ingerer_meteo

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")

load_dotenv(dotenv_path="../.env")

client_s3 = boto3.client(
    "s3",
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY"),
    region_name=os.environ.get("AWS_REGION")
)

ROOT_PATH = Path.cwd().resolve().parent

# Charger le fichier poi.parquet le plus récent
poi_df = pl.read_parquet(ROOT_PATH / "data" / "processed" / "2026-07-25_150301_poi.parquet")

poi_sample = poi_df  # tous les POI, plus d'échantillonnage

correspondance_poi_fichier = {}

for row in poi_sample.iter_rows(named=True):
    nom_fichier = ingerer_meteo(
        latitude=row["latitude"],
        longitude=row["longitude"],
        start_date="2026-07-20",
        end_date="2026-07-24",
        hourly="temperature_2m",
        identifiant=f"poi{row['poi_id']}",
        bucket_s3="electric-mobility-platform-thierry",
        client_s3=client_s3,
        root_path=ROOT_PATH,
    )
    if nom_fichier is not None:
        correspondance_poi_fichier[row["poi_id"]] = nom_fichier

correspondance_poi_fichier

2026-08-03 16:40:19,617 - ingestion.meteo - INFO - Requête Open-Meteo effectuée avec succès.
2026-08-03 16:40:19,620 - common.io - INFO - Données JSON sauvegardées en local : /home/thierry/code/thcarole1/projects/electric-mobility-platform/data/raw/2026-08-03_164019_poi7008_meteo.json
2026-08-03 16:40:19,960 - common.io - INFO - Upload réussi du fichier /home/thierry/code/thcarole1/projects/electric-mobility-platform/data/raw/2026-08-03_164019_poi7008_meteo.json
2026-08-03 16:40:20,157 - ingestion.meteo - INFO - Requête Open-Meteo effectuée avec succès.
2026-08-03 16:40:20,160 - common.io - INFO - Données JSON sauvegardées en local : /home/thierry/code/thcarole1/projects/electric-mobility-platform/data/raw/2026-08-03_164019_poi6931_meteo.json
2026-08-03 16:40:20,262 - common.io - INFO - Upload réussi du fichier /home/thierry/code/thcarole1/projects/electric-mobility-platform/data/raw/2026-08-03_164019_poi6931_meteo.json
2026-08-03 16:40:20,444 - ingestion.meteo - INFO - Requête Open-Me

{7008: '2026-08-03_164019_poi7008_meteo.json',
 6931: '2026-08-03_164019_poi6931_meteo.json',
 60368: '2026-08-03_164020_poi60368_meteo.json',
 6977: '2026-08-03_164020_poi6977_meteo.json',
 7025: '2026-08-03_164023_poi7025_meteo.json',
 198809: '2026-08-03_164026_poi198809_meteo.json',
 198838: '2026-08-03_164027_poi198838_meteo.json',
 7026: '2026-08-03_164027_poi7026_meteo.json',
 198846: '2026-08-03_164027_poi198846_meteo.json',
 7024: '2026-08-03_164028_poi7024_meteo.json',
 198835: '2026-08-03_164028_poi198835_meteo.json',
 198815: '2026-08-03_164028_poi198815_meteo.json',
 77520: '2026-08-03_164029_poi77520_meteo.json',
 6928: '2026-08-03_164029_poi6928_meteo.json',
 198842: '2026-08-03_164029_poi198842_meteo.json',
 198862: '2026-08-03_164029_poi198862_meteo.json',
 198823: '2026-08-03_164030_poi198823_meteo.json',
 198787: '2026-08-03_164030_poi198787_meteo.json',
 198782: '2026-08-03_164030_poi198782_meteo.json',
 198871: '2026-08-03_164031_poi198871_meteo.json',
 198777: '20